In [ ]:
# Import required libraries
import os
import pymupdf
from openai import OpenAI
# from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display

In [ ]:
# load_dotenv(override=True)

MODEL = "ollama3.2"
# OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
ollama = OpenAI(api_key="ollama", base_url="http:localhost:11434/v1")

In [ ]:
def pdf_text_extractor(pdf_path: str) -> str:
    """
    Extract all the context of a PDF file as text.

    Parameters:
        pdf_path (str): PDF file path.

    Returns:
        str: Context extracted from PDF file as text.
    """
    doc = pymupdf.open(pdf_path)
    text = ""
    # Extract text from each page of PDF file
    text += "\n".join(page.get_text() for page in doc)
    # Close PDF file
    doc.close()
    return text

In [ ]:
PDF_DOC = "sample.pdf"
pdf_content = pdf_text_extractor(PDF_DOC)

SYSTEM_PROMPT = f"""
You are a professional content summarizer. First analyze the content provided to you carfully and then \
keep the most important parts as a summary. Respond in Markdown - Do not wrap Markdown in a code block. \

Here is the content:
{pdf_content}

Respond just with Markdown.
"""

message = [
    {"role": "system", "content": SYSTEM_PROMPT}
]

In [ ]:
# Call Ollama api, get the response and show the result
response = ollama.chat.completions.create(model=MODEL, messages=message)
llm_response = response.choices[0].message.content
print(llm_response)

In [ ]:
# Show LLM response in Markdown
display(Markdown(llm_response))

In [ ]:
def stream_summary():
    """
    Stream LLM (Ollama) answer instead of printing result at once.
    """
    stream = ollama.chat.completions.create(
        model=MODEL,
        messagegs=message,
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        update_display(display(Markdown(response), display_id=display_handle.display_id))

In [ ]:
stream_summary()